# Decision Tree Benchmarking

In this notebook, we compare our custom-built Decision Tree classifier (using Entropy and Information Gain) against the `scikit-learn` implementation. We will evaluate how well it handles continuous attributes on the Breast Cancer dataset.

In [1]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import numpy as np
from time import time
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier as SklearnDT

# Import our custom modules
from classical_ml.tree_based.decision_tree import DecisionTree as CustomDT
from utils.metrics import accuracy_score, precision_score, recall_score, f1_score

d:\Github\Classical-ML-From-Scratch\classical_ml\tree_based\decision_tree.py:76: SyntaxWarning: invalid escape sequence '\s'
  Formula: Gain(S, A) = Entropy(S) - \sum (|Sv| / |S|) * Entropy(Sv)
d:\Github\Classical-ML-From-Scratch\classical_ml\tree_based\decision_tree.py:104: SyntaxWarning: invalid escape sequence '\s'
  Formula: -\sum p * log2(p)


In [2]:
# 1. Load Dataset
data = load_breast_cancer()
X, y = data.data, data.target

# 2. Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Note: Decision Trees are scale-invariant, meaning they don't strictly require 
# standardized data. The Information Gain calculation simply looks for the best 
# continuous threshold regardless of the absolute scale.
print(f"Training data shape: {X_train.shape}")
print(f"Testing data shape: {X_test.shape}")

Training data shape: (455, 30)
Testing data shape: (114, 30)


In [3]:
print("1. Custom Decision Tree")
start = time()

# Initialize Decision Tree with max depth to prevent severe overfitting
custom_dt = CustomDT(max_depth=10)
custom_dt.fit(X_train, y_train)
preds_custom = custom_dt.predict(X_test)
time_custom = time() - start

print(f"Accuracy  : {accuracy_score(y_test, preds_custom):.4f}")
print(f"Precision : {precision_score(y_test, preds_custom):.4f}")
print(f"Recall    : {recall_score(y_test, preds_custom):.4f}")
print(f"F1-Score  : {f1_score(y_test, preds_custom):.4f}")
print(f"Time Taken: {time_custom:.5f} seconds\n")

1. Custom Decision Tree
Accuracy  : 0.9386
Precision : 0.9444
Recall    : 0.9577
F1-Score  : 0.9510
Time Taken: 2.85708 seconds



In [6]:
print("2. Scikit-Learn Decision Tree")
start = time()

# We set criterion='entropy' to match our custom mathematical formulation
# We also set a fixed random_state for reproducibility
sk_dt = SklearnDT(criterion='entropy', max_depth=10, random_state=42)
sk_dt.fit(X_train, y_train)
preds_sk = sk_dt.predict(X_test)
time_sk = time() - start

print(f"Accuracy  : {accuracy_score(y_test, preds_sk):.4f}")
print(f"Precision : {precision_score(y_test, preds_sk):.4f}")
print(f"Recall    : {recall_score(y_test, preds_sk):.4f}")
print(f"F1-Score  : {f1_score(y_test, preds_sk):.4f}")
print(f"Time Taken: {time_sk:.5f} seconds\n")

2. Scikit-Learn Decision Tree
Accuracy  : 0.9474
Precision : 0.9333
Recall    : 0.9859
F1-Score  : 0.9589
Time Taken: 0.00866 seconds



## Conclusion
Both models successfully utilize Information Gain derived from Entropy to recursively split the dataset. 

Because finding the optimal continuous threshold requires iterating through all unique values of every feature at each node split, the training time for a pure Python implementation is expectedly higher than `scikit-learn`, which uses highly optimized Cython routines. Additionally, differences in the exact accuracy score are normal because Decision Trees are heavily sensitive to small data variations and how tie-breaking is handled internally during the split evaluation.